# ML-07 — Baseline Action Score and Top-20 Review

Build a readable rule that ranks content pages for human review. This is a decision-support queue, not a prediction of Google's ranking system. We will evaluate its ranking against the notebook's decline label, while keeping that label out of the score.

## 1. My rule and its reason codes

**Rule:** prioritize pages that have meaningful search visibility, have gone longer without an update, have a known search position, or appear short relative to visible demand. The score is a fixed weighted sum of four transparent components: 40% visibility, 30% update age, 25% search-position opportunity, and 5% content-depth gap. There are no fitted weights.

Reason codes describe review evidence, not the label: `stale_visible_page` (not updated in 180+ days and 500+ impressions), `thin_visible_page` (known word count below 1,200 and 250+ impressions), `low_ctr_visible_page` (CTR below 0.5%, 500+ impressions, known position within 20), and `search_position_opportunity` (position 4–20 and 500+ impressions). Pages with none of these reasons receive `general_refresh_review`.

The position component rises from rank 1 toward rank 20, where an already-visible page may have room to improve. The score uses neither `trend_direction` nor `trend_pct`; those define the target. IDs are for identifying rows and grouping the holdout only.

In [1]:
from pathlib import Path
import numpy as np
import pandas as pd
from IPython.display import display
from sklearn.model_selection import train_test_split

DATA_FILENAME = 'content_refresh_anonymized.csv'
OUTPUT_RELATIVE_PATH = Path('work') / 'outputs' / 'baseline_action_score.csv'
RANDOM_STATE = 42

# Locate the repository root from the notebook's current working directory.
repo_root = next(
    (parent for parent in (Path.cwd(), *Path.cwd().parents)
     if (parent / 'data' / 'raw' / DATA_FILENAME).is_file()),
    None,
)
if repo_root is None:
    raise FileNotFoundError(
        f'Could not find data/raw/{DATA_FILENAME} above {Path.cwd()}. '
        'Open the notebook from the repository or place the anonymized starter CSV there.'
    )

data_path = repo_root / 'data' / 'raw' / DATA_FILENAME
output_path = repo_root / OUTPUT_RELATIVE_PATH

required_columns = {
    'content_id', 'client_id', 'trend_direction', 'impressions_90d',
    'days_since_last_update', 'avg_position', 'word_count', 'ctr',
    'content_age_days',
}
frame = pd.read_csv(data_path)
missing_columns = sorted(required_columns.difference(frame.columns))
if missing_columns:
    raise ValueError(f'Required columns are missing: {missing_columns}')
if frame.empty:
    raise ValueError(f'The input file is empty: {data_path}')

# Match the upstream prepared cohort: measurable search visibility and content at least 90 days old.
frame = frame.loc[
    pd.to_numeric(frame['impressions_90d'], errors='coerce').gt(0)
    & pd.to_numeric(frame['content_age_days'], errors='coerce').ge(90)
].drop_duplicates(subset=['content_id']).reset_index(drop=True)
if frame.empty:
    raise ValueError('No rows remain after applying the prepared-cohort filters.')

numeric_columns = [
    'impressions_90d', 'days_since_last_update', 'avg_position',
    'word_count', 'ctr', 'content_age_days',
]
for column in numeric_columns:
    frame[column] = pd.to_numeric(frame[column], errors='coerce')

if frame['impressions_90d'].isna().any():
    raise ValueError('impressions_90d contains missing or non-numeric values after filtering.')
if frame['trend_direction'].isna().any():
    raise ValueError('trend_direction contains missing values; the evaluation label cannot be defined.')

# This label is only for retrospective evaluation; it is never used to calculate the score.
frame['is_declining_label'] = frame['trend_direction'].astype(str).str.lower().eq('down').astype('int8')
if frame['is_declining_label'].nunique() < 2:
    raise ValueError('The evaluation target has only one class; Precision@K is not informative.')

SCORING_INPUTS = [
    'impressions_90d', 'days_since_last_update', 'avg_position', 'word_count',
]
FORBIDDEN_SCORE_INPUTS = {
    'trend_direction', 'trend_pct', 'is_declining_label',
    'impressions_last_30d', 'impressions_prev_30d',
    'has_clicks', 'has_ai_sessions', 'measurable_opportunity',
    'content_id', 'client_id',
}
DECISION_INPUTS = SCORING_INPUTS + ['ctr']  # CTR is used only for a reason code.
assert not set(DECISION_INPUTS).intersection(FORBIDDEN_SCORE_INPUTS)


def percentile_rank(values: pd.Series, *, missing_value: float = 0.5) -> pd.Series:
    return values.rank(method='average', pct=True).fillna(missing_value)


def reason_codes(row: pd.Series) -> list[str]:
    reasons = []
    impressions = row['impressions_90d']
    position = row['avg_position']
    if pd.notna(row['days_since_last_update']) and row['days_since_last_update'] >= 180 and impressions >= 500:
        reasons.append('stale_visible_page')
    if pd.notna(row['word_count']) and 0 < row['word_count'] < 1200 and impressions >= 250:
        reasons.append('thin_visible_page')
    if pd.notna(row['ctr']) and pd.notna(position) and 0 < position <= 20 and impressions >= 500 and row['ctr'] < 0.5:
        reasons.append('low_ctr_visible_page')
    if pd.notna(position) and 4 <= position <= 20 and impressions >= 500:
        reasons.append('search_position_opportunity')
    return reasons or ['general_refresh_review']


def suggested_action(reasons: list[str]) -> str:
    if 'thin_visible_page' in reasons:
        return 'review_content_depth'
    if 'low_ctr_visible_page' in reasons:
        return 'review_title_and_snippet'
    if 'stale_visible_page' in reasons or 'search_position_opportunity' in reasons:
        return 'review_for_refresh'
    return 'monitor_and_review'

print(f'Loaded {len(frame):,} eligible rows from the anonymized starter data.')
print(f'Input file: {data_path.relative_to(repo_root)}')
print(f'Target prevalence in this cohort: {frame["is_declining_label"].mean():.3f} (evaluation only)')

Loaded 30,000 eligible rows from the anonymized starter data.
Input file: data\raw\content_refresh_anonymized.csv
Target prevalence in this cohort: 0.542 (evaluation only)


## 2. Build the ranked queue (writes the CSV)

Create a score and reason codes for every eligible row, export the queue without the evaluation label, then measure Precision@K on the held-out clients. The split below mirrors the deterministic split in `scripts/03_train_model.py` (seed 42); if it cannot hold out clients with both label classes, it uses the same stratified-row fallback.

In [2]:
log_impressions = np.log1p(frame['impressions_90d'])
visibility_score = percentile_rank(log_impressions, missing_value=0.0)

update_days = frame['days_since_last_update']
update_days_for_score = update_days.fillna(update_days.median())
freshness_risk_score = percentile_rank(update_days_for_score, missing_value=0.0)

position = frame['avg_position'].where(frame['avg_position'].gt(0))
position_opportunity_score = (
    ((position.clip(lower=1, upper=20) - 1) / 19).fillna(0.0)
    * visibility_score
    * position.between(1, 20).astype(float)
)

word_count = frame['word_count'].where(frame['word_count'].gt(0))
word_count_rank = percentile_rank(word_count, missing_value=0.5)
depth_gap_score = (1 - word_count_rank) * visibility_score

frame['visibility_score'] = visibility_score
frame['freshness_risk_score'] = freshness_risk_score
frame['position_opportunity_score'] = position_opportunity_score
frame['depth_gap_score'] = depth_gap_score
frame['baseline_action_score'] = (
    0.40 * frame['visibility_score']
    + 0.30 * frame['freshness_risk_score']
    + 0.25 * frame['position_opportunity_score']
    + 0.05 * frame['depth_gap_score']
).clip(0, 1)

frame['reason_codes'] = frame.apply(lambda row: '|'.join(reason_codes(row)), axis=1)
frame['suggested_action'] = frame['reason_codes'].apply(
    lambda value: suggested_action(value.split('|'))
)
frame['baseline_rank'] = frame['baseline_action_score'].rank(
    method='first', ascending=False
).astype(int)

queue_columns = [
    'content_id', 'client_id', 'baseline_rank', 'baseline_action_score',
    'visibility_score', 'freshness_risk_score', 'position_opportunity_score',
    'depth_gap_score', 'reason_codes', 'suggested_action',
    'impressions_90d', 'days_since_last_update', 'avg_position',
    'word_count', 'ctr',
]
queue = frame[queue_columns].sort_values('baseline_rank').reset_index(drop=True)
output_path.parent.mkdir(parents=True, exist_ok=True)
queue.to_csv(output_path, index=False)

# Reproduce the ML-08 client-holdout assignment exactly.
all_indices = np.arange(len(frame))
client_series = frame['client_id'].fillna('unknown').astype(str)
unique_clients = client_series.drop_duplicates().to_numpy()
if len(unique_clients) >= 5:
    shuffled_clients = np.random.default_rng(RANDOM_STATE).permutation(unique_clients)
    test_client_count = max(1, int(round(len(shuffled_clients) * 0.2)))
    test_clients = set(shuffled_clients[:test_client_count])
    test_mask = client_series.isin(test_clients).to_numpy()
    train_indices = all_indices[~test_mask]
    test_indices = all_indices[test_mask]
    split_strategy = 'client_holdout'
    if (
        len(train_indices) == 0 or len(test_indices) == 0
        or frame.iloc[train_indices]['is_declining_label'].nunique() < 2
        or frame.iloc[test_indices]['is_declining_label'].nunique() < 2
    ):
        train_indices, test_indices = train_test_split(
            all_indices, test_size=0.2, random_state=RANDOM_STATE,
            stratify=frame['is_declining_label'],
        )
        train_indices = np.asarray(train_indices)
        test_indices = np.asarray(test_indices)
        split_strategy = 'stratified_row_holdout'
else:
    train_indices, test_indices = train_test_split(
        all_indices, test_size=0.2, random_state=RANDOM_STATE,
        stratify=frame['is_declining_label'],
    )
    train_indices = np.asarray(train_indices)
    test_indices = np.asarray(test_indices)
    split_strategy = 'stratified_row_holdout'


def precision_at_k(labels: pd.Series, scores: pd.Series, k: int) -> float:
    ranked = pd.DataFrame({'label': labels.to_numpy(), 'score': scores.to_numpy()})
    top = ranked.sort_values('score', ascending=False).head(min(k, len(ranked)))
    return float(top['label'].mean()) if len(top) else 0.0


test_rows = frame.iloc[test_indices]
test_labels = test_rows['is_declining_label']
ks = [k for k in (20, 50, 100) if len(test_rows) >= k]
metric_rows = [
    {'metric': 'held-out base rate', 'value': float(test_labels.mean())},
]
metric_rows.extend(
    {
        'metric': f'baseline Precision@{k}',
        'value': precision_at_k(test_labels, test_rows['baseline_action_score'], k),
    }
    for k in ks
)
metrics = pd.DataFrame(metric_rows)

print(f'Split: {split_strategy}; train rows: {len(train_indices):,}; held-out rows: {len(test_indices):,}')
print(f'Wrote {len(queue):,} ranked rows to {output_path.relative_to(repo_root)}')
display(metrics.style.format({'value': '{:.3f}'}))

Split: client_holdout; train rows: 27,675; held-out rows: 2,325
Wrote 30,000 ranked rows to work\outputs\baseline_action_score.csv


,metric,value
0,held-out base rate,0.391
1,baseline Precision@20,0.400
2,baseline Precision@50,0.340
3,baseline Precision@100,0.370


## 3. Top-20 review

A reason code explains why the rule surfaced a page. Confidence is deliberately described as a heuristic, not a calibrated probability. The final column lists a concrete reason the recommendation could be wrong; these are review prompts, not conclusions.

In [3]:
top20 = queue.head(20).copy()

def confidence_note(reason_string: str) -> str:
    count = len(reason_string.split('|'))
    if reason_string == 'general_refresh_review':
        return 'Limited: continuous score only; inspect manually.'
    if count >= 2:
        return 'More evidence: multiple rule triggers; still heuristic.'
    return 'Some evidence: one threshold trigger; verify context.'


def what_could_make_it_wrong(row: pd.Series) -> str:
    reasons = set(row['reason_codes'].split('|'))
    if 'thin_visible_page' in reasons:
        return 'Word count may be missing or short by design; length alone does not establish quality.'
    if 'low_ctr_visible_page' in reasons:
        return 'Average CTR can hide query mix, device, and position effects; inspect context before editing.'
    if 'stale_visible_page' in reasons:
        return 'An old update date does not prove the page is inaccurate or needs revision.'
    if 'search_position_opportunity' in reasons:
        return 'Average position can hide query-level variation; the page may already satisfy its intent.'
    return 'The score is driven by broad proxy signals and may not represent business value.'


top20['confidence_note'] = top20['reason_codes'].apply(confidence_note)
top20['what_could_make_it_wrong'] = top20.apply(what_could_make_it_wrong, axis=1)
review_columns = [
    'baseline_rank', 'suggested_action', 'reason_codes', 'baseline_action_score',
    'confidence_note', 'what_could_make_it_wrong',
]
display(top20[review_columns].style.format({'baseline_action_score': '{:.3f}'}))

,baseline_rank,suggested_action,reason_codes,baseline_action_score,confidence_note,what_could_make_it_wrong
0,1,review_title_and_snippet,stale_visible_page|low_ctr_visible_page|search_position_opportunity,0.942,More evidence: multiple rule triggers; still heuristic.,"Average CTR can hide query mix, device, and position effects; inspect context before editing."
1,2,review_title_and_snippet,low_ctr_visible_page|search_position_opportunity,0.892,More evidence: multiple rule triggers; still heuristic.,"Average CTR can hide query mix, device, and position effects; inspect context before editing."
2,3,review_title_and_snippet,low_ctr_visible_page|search_position_opportunity,0.889,More evidence: multiple rule triggers; still heuristic.,"Average CTR can hide query mix, device, and position effects; inspect context before editing."
3,4,review_title_and_snippet,low_ctr_visible_page|search_position_opportunity,0.888,More evidence: multiple rule triggers; still heuristic.,"Average CTR can hide query mix, device, and position effects; inspect context before editing."
4,5,review_for_refresh,search_position_opportunity,0.885,Some evidence: one threshold trigger; verify context.,Average position can hide query-level variation; the page may already satisfy its intent.
5,6,review_title_and_snippet,low_ctr_visible_page|search_position_opportunity,0.884,More evidence: multiple rule triggers; still heuristic.,"Average CTR can hide query mix, device, and position effects; inspect context before editing."
6,7,review_title_and_snippet,low_ctr_visible_page|search_position_opportunity,0.884,More evidence: multiple rule triggers; still heuristic.,"Average CTR can hide query mix, device, and position effects; inspect context before editing."
7,8,review_for_refresh,search_position_opportunity,0.883,Some evidence: one threshold trigger; verify context.,Average position can hide query-level variation; the page may already satisfy its intent.
8,9,review_title_and_snippet,low_ctr_visible_page|search_position_opportunity,0.882,More evidence: multiple rule triggers; still heuristic.,"Average CTR can hide query mix, device, and position effects; inspect context before editing."
9,10,review_title_and_snippet,low_ctr_visible_page|search_position_opportunity,0.881,More evidence: multiple rule triggers; still heuristic.,"Average CTR can hide query mix, device, and position effects; inspect context before editing."


## 4. Weak picks + leakage check

A top-ranked item whose target label is 0 is a **label disagreement**, not automatically a bad recommendation: the label measures recent impression decline, while this queue prioritizes review opportunity. Use disagreements and missing evidence to decide what a human should inspect. The check also confirms that the score did not use label-derived trend fields, product flags, IDs, or the two component windows used to construct the label.

In [4]:
decision_inputs = set(DECISION_INPUTS)
forbidden_inputs = {
    'trend_direction', 'trend_pct', 'is_declining_label',
    'impressions_last_30d', 'impressions_prev_30d',
    'has_clicks', 'has_ai_sessions', 'measurable_opportunity',
    'content_id', 'client_id',
}
assert decision_inputs.isdisjoint(forbidden_inputs)
assert frame['baseline_action_score'].notna().all()
assert frame['baseline_action_score'].between(0, 1).all()
assert queue['baseline_rank'].is_monotonic_increasing

# Label disagreement is a diagnostic prompt only; it is not ground truth for refresh value.
top20_with_label = queue.head(20).merge(
    frame[['content_id', 'is_declining_label']],
    on='content_id',
    how='left',
    validate='one_to_one',
)
label_disagreements = top20_with_label.loc[
    top20_with_label['is_declining_label'].eq(0),
    ['baseline_rank', 'reason_codes', 'is_declining_label'],
]

print(f'Leakage check passed. Decision inputs: {sorted(decision_inputs)}')
print(
    'No trend label fields, direct 30-day comparison windows, product flags, or IDs '
    'enter the score.'
)
print(
    f'Top-20 label disagreements: {len(label_disagreements)}. '
    'These deserve review; they are not proven false positives.'
)
display(label_disagreements.head(5))

top20_reason_counts = top20['reason_codes'].str.split('|').explode().value_counts()
print('Reason-code counts in the top 20:')
display(top20_reason_counts.rename('count').to_frame())
if top20_reason_counts.get('search_position_opportunity', 0) >= 0.75 * len(top20):
    print(
        'Weak-pick finding: the top of the queue is concentrated in one position pattern. '
        'This may be a repeated hypothesis, not many independent opportunities; inspect '
        'query-level context before acting.'
    )

print(
    'Interpretation limit: the 90-day snapshot features overlap the period used to derive '
    'the 30-day trend label. This is retrospective prioritization, not a forecast of future decline.'
)

Leakage check passed. Decision inputs: ['avg_position', 'ctr', 'days_since_last_update', 'impressions_90d', 'word_count']
No trend label fields, direct 30-day comparison windows, product flags, or IDs enter the score.
Top-20 label disagreements: 10. These deserve review; they are not proven false positives.


,baseline_rank,reason_codes,is_declining_label
2,3,low_ctr_visible_page|search_position_opportunity,0
4,5,search_position_opportunity,0
5,6,low_ctr_visible_page|search_position_opportunity,0
6,7,low_ctr_visible_page|search_position_opportunity,0
7,8,search_position_opportunity,0


Reason-code counts in the top 20:


,count
reason_codes,
search_position_opportunity,20
low_ctr_visible_page,16
stale_visible_page,1


Weak-pick finding: the top of the queue is concentrated in one position pattern. This may be a repeated hypothesis, not many independent opportunities; inspect query-level context before acting.
Interpretation limit: the 90-day snapshot features overlap the period used to derive the 30-day trend label. This is retrospective prioritization, not a forecast of future decline.


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.